# Linear Algebra Primer

State space modeling is based on linear algebra. Here, we will explore a bit how we can use SymPy for Linear Algebra.


In [1]:
from IPython.core.display import HTML
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 10,
})

from sympy import *

from mathprint import *

## 1. Matrices and Vectors in SymPy

A state vector $\mathbf{x}$ is just a column matrix; $A$, $B$, $C$, $D$ are
matrices built from symbols or numbers. SymPy represents all of these with
`Matrix`.


In [2]:
x1, x2 = symbols('x1 x2')

x = Matrix([[x1], [x2]])           # state vector (2x1)
A = Matrix([[0, 1], [-2, -3]])     # 2x2 numeric matrix
I2 = eye(2)                        # identity
Z2 = zeros(2, 2)                   # zero matrix

mprint("x=",latex(x))
mprint("A=",latex(A))
mprint("I_2=",latex(I2))
mprint("Z_2=",latex(Z2))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Matrix multiplication, addition, and transpose work as expected:

In [3]:
y1 = A * x
mprint("y_1=",latex(y1))

y2 = A + I2
mprint("y_2=",latex(y2))

y3 = A.T
mprint("y_3=",latex(y3))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. Matrix Inverse and Rank

The **inverse** $A^{-1}$ exists only when $\det(A) \neq 0$ — i.e. when $A$'s
columns (or rows) are linearly independent. The **rank** of a matrix is the
number of linearly independent rows (equivalently, columns) it has.

Rank is the tool we will later use to test controllability and
observability: a "full rank" matrix means every direction is reachable
(or every state is distinguishable from the output); a rank-deficient one
means some direction is invisible or unreachable.


**Full-rank matrix example:**

In [4]:
A_full = Matrix([[1, 2], [3, 4]])

print("A_full:")
mprint(latex(A_full))
print("det(A_full):")
mprint(latex(A_full.det()))
print("rank(A_full):")
mprint(latex(A_full.rank()))
print("inv(A_full):")
mprint(latex(A_full.inv()))



A_full:


<IPython.core.display.Math object>

det(A_full):


<IPython.core.display.Math object>

rank(A_full):


<IPython.core.display.Math object>

inv(A_full):


<IPython.core.display.Math object>

**Rank-deficient matrix example:**

In [22]:
# A rank-deficient matrix: second row is a multiple of the first
A_deficient = Matrix([[1, 2], [2, 4]])

print("A_deficient:")
mprint(latex(A_deficient))
print("det(A_deficient):")
mprint(latex(A_deficient.det()))
print("rank(A_deficient):")
mprint(latex(A_deficient.rank()))

A_deficient:


<IPython.core.display.Math object>

det(A_deficient):


<IPython.core.display.Math object>

rank(A_deficient):


<IPython.core.display.Math object>

Because $\det(A_{\text{deficient}}) = 0$, this matrix has no
inverse, thus, calling `.inv()` on it raises an error. Geometrically, both rows
point along the same direction, so the matrix collapses the plane onto a
line instead of spanning it.


## 3. Eigenvalues and the Characteristic Polynomial

An eigenvalue $\lambda$ and eigenvector $\mathbf{v} \neq 0$ of $A$ satisfy

$$
A\mathbf{v} = \lambda \mathbf{v}
$$

Equivalently, $\lambda$ is a root of the **characteristic polynomial**

$$
\det(\lambda I - A) = 0
$$


In [6]:
A = Matrix([[0, 1], [-2, -3]])
A.eigenvals()


{-1: 1, -2: 1}

This is the same expression that appears throughout the earlier chapters
of this book as $\det(sI - A)$ in a transfer function's denominator. **The
eigenvalues of $A$ are exactly the poles of the system**. 

In [7]:
s = symbols('s')
charpoly = A.charpoly(s).as_expr()
print("--- Characteristic Polynomial: ---")
mprint(latex(charpoly))
print("--- Factored: ---")
mprint(latex(factor(charpoly)))
print("--- Eigenvalues: ---")
mprint(latex(solve(charpoly, s)))

--- Characteristic Polynomial: ---


<IPython.core.display.Math object>

--- Factored: ---


<IPython.core.display.Math object>

--- Eigenvalues: ---


<IPython.core.display.Math object>

As expected, the roots of the characteristic polynomial
($-1, -2$) match the eigenvalues computed directly above.


## 4. Similarity Transforms

Two matrices $A$ and $A' = T^{-1}AT$ (for any invertible $T$) are called
**similar**. Similar matrices represent the *same* linear map in different
coordinates. **Similar matrices have identical
eigenvalues**.

This is why the canonical-form state-space model and the physical-
coordinate state-space model of the same system (Sections 1 and 2 of
*State-Space Modeling*) can look completely different yet describe
identical dynamics: they're related by a similarity transform, a change of
basis for the state vector, not a change in the system itself.


Example:
Find similar matrix to $A$ for the ginven similarity transform matrix $T$!

In [8]:
A = Matrix([[0, 1], [-2, -3]])
T = Matrix([[1, 1], [0, 1]])   # any invertible change of basis

mprint("A=",latex(A))
mprint("T=",latex(T))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [9]:
T = Matrix([[1, 1], [0, 1]])   # any invertible change of basis
A_prime = T.inv() * A * T

mprint("A'=",latex(A_prime))
print("--- Eigenvalues of A: ---")
mprint(latex(A.eigenvals()))
print("--- Eigenvalues of A': ---")
mprint(latex(simplify(A_prime).eigenvals()))


<IPython.core.display.Math object>

--- Eigenvalues of A: ---


<IPython.core.display.Math object>

--- Eigenvalues of A': ---


<IPython.core.display.Math object>

Same eigenvalues, different-looking matrix!


## 5. The Matrix Exponential

For a scalar ODE $\dot{x} = ax$, the solution is $x(t) = e^{at}x(0)$. The
matrix generalization is identical in form:

$$
\dot{\mathbf{x}} = A\mathbf{x} \quad\Longrightarrow\quad
\mathbf{x}(t) = e^{At}\mathbf{x}(0)
$$

where $e^{At}$ is defined by the same Taylor series as the scalar
exponential, just with matrix powers:

$$
e^{At} = I + At + \frac{(At)^2}{2!} + \frac{(At)^3}{3!} + \cdots
$$

SymPy computes this directly for numeric matrices.


In [10]:
mprint("A=",latex(A))
t = symbols('t', positive=True)

print("--- Matrix Exponential: ---")
expA = (A * t).exp()
mprint("e^{At}=",latex(expA))


<IPython.core.display.Math object>

--- Matrix Exponential: ---


<IPython.core.display.Math object>

Differentiating this result with respect to $t$ and multiplying by
$A$ should reproduce it exactly — confirming $\frac{d}{dt}e^{At} =
Ae^{At}$, which is exactly why $\mathbf{x}(t) = e^{At}\mathbf{x}(0)$ solves
$\dot{\mathbf{x}} = A\mathbf{x}$.


In [11]:
lhs = diff(expA, t)
rhs = A * expA
mprint("d/dt e^{At}=",latex(lhs))
mprint("A e^{At}=",latex(rhs))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 6. The Cayley-Hamilton Theorem

Every matrix satisfies its own characteristic polynomial. If

$$
\phi(\lambda) = \det(\lambda I - A) = \lambda^n + c_{n-1}\lambda^{n-1} + \cdots + c_0
$$

then

$$
\phi(A) = A^n + c_{n-1}A^{n-1} + \cdots + c_0 I = 0
$$

Substituting the matrix $A$ itself into its own characteristic
polynomial always gives the zero matrix. 

This looks like a strange thing to do (plugging a matrix into a polynomial that was built *from* that
matrix), but it's exactly the mechanism Ackermann's formula uses later to
compute state-feedback gains: evaluating a *desired* characteristic
polynomial at $A$ is the same operation, just with different coefficients.


In [12]:
n = A.shape[0]
charpoly_expr = A.charpoly(s).as_expr() # get the characteristic polynomial as an expression
poly = Poly(charpoly_expr, s)
coeffs = poly.all_coeffs()  # highest degree first

phi_A = zeros(n, n)
deg = len(coeffs) - 1
str = ''
for i, c in enumerate(coeffs):
    power = deg - i
    phi_A += c * A**power
    str += f"{latex(c)} A^{power} + "
str = str.rstrip(" + ")
mprint(f"\\text{{Characteristic Polynomial}}: {latex(charpoly_expr)}")
mprint(f"\\text{{Cayley-Hamilton Theorem}}: {latex(phi_A)} = 0")
mprint(f"\\text{{Cayley-Hamilton Theorem (expanded)}}: {str}= 0")



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>